In [ ]:
import pandas as pd
import joblib
import shap
import matplotlib.pyplot as plt

# 1. Cargar datos procesados y modelo supervisado
df = pd.read_csv('../data/datacloud_churn_clean.csv')
model = joblib.load('../models/churn_model_rf.pkl')
features = joblib.load('../models/model_features.pkl')

X = df[features]

# 2. Inicializar Explainer de SHAP
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X)

# Manejo de dimensiones de shap_values según versión
if isinstance(shap_values, list):
    vals = shap_values[1]  # Clase positiva (Churn)
else:
    vals = shap_values[:, :, 1] if len(shap_values.shape) == 3 else shap_values

# 3. Gráfico Global de Importancia (Summary Plot)
plt.figure(figsize=(10, 6))
shap.summary_plot(vals, X, plot_type="bar", show=False)
plt.title("Top Variables Determinantes en el Churn (SHAP Global)")
plt.tight_layout()
plt.show()

# 4. Explicación Individual para 2 Clientes Específicos (Caso 1: Bajo riesgo, Caso 2: Alto riesgo)
print("=== EXPLICACIÓN INDIVIDUAL: CLIENTE 0 vs CLIENTE 5 ===")
plt.figure(figsize=(10, 4))
shap.plots.force(explainer.expected_value[1], vals[0], X.iloc[0], matplotlib=True)
plt.title("Factores de Riesgo - Cliente 0")
plt.show()

plt.figure(figsize=(10, 4))
shap.plots.force(explainer.expected_value[1], vals[5], X.iloc[5], matplotlib=True)
plt.title("Factores de Riesgo - Cliente 5")
plt.show()

c:\Users\jean9\TCC_Churn_DataCloud\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Análisis de Paridad Demográfica (Fairness)
df['Prob_Churn'] = model.predict_proba(X)[:, 1]
df['Pred_Churn'] = (df['Prob_Churn'] >= 0.35).astype(int)

print("=== TASA DE PREDICCIÓN DE CHURN POR GÉNERO ===")
print(df.groupby('Genero_M')['Pred_Churn'].mean().rename({0: 'Femenino/Otro', 1: 'Masculino'}))

print("\n=== TASA DE PREDICCIÓN DE CHURN POR REGIÓN ===")
regiones = [col for col in df.columns if col.startswith('Region_')]
for reg in regiones:
    tasa = df[df[reg] == 1]['Pred_Churn'].mean()
    print(f"{reg}: {tasa:.2%}")